# Stage1 paper forward readiness audit
Read-only sealed code and native Qlib source checks.


In [1]:
import ast,datetime,hashlib,json,re,subprocess,sys
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
import pandas as pd,qlib
from qlib.data import D
from src.data.nas_market_data import _settings,_latest_verified_snapshot
card=root/'research/stage1_paper_forward_readiness_audit_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
now=datetime.datetime.now(datetime.timezone.utc);assert plan['frozen'] and datetime.datetime.fromisoformat(plan['as_of_utc_window'][0].replace('Z','+00:00'))<=now<=datetime.datetime.fromisoformat(plan['as_of_utc_window'][1].replace('Z','+00:00'))
assert sha(root/src['release_manifest'])==src['release_manifest_sha256'] and sha(root/src['provider_uri']/'qlib_store_manifest.json')==src['provider_manifest_sha256']
assert sha(root/'src/workflow/stage1_open_action_value.py')==src['current_open_action_sha256'] and sha(root/'src/workflow/intraday_recovery_direct_allocation.py')==src['current_recovery_loader_sha256']
manifest=json.loads((root/src['release_manifest']).read_text());provider_manifest=json.loads((root/src['provider_uri']/'qlib_store_manifest.json').read_text());files=manifest['code_sha256'];mismatches=[]
for rel,expected in files.items():
 current=sha(root/rel)
 if current!=expected:mismatches.append({'path':rel,'sealed_sha256':expected,'current_sha256':current})
assert len(files)==12 and set(x['path'] for x in mismatches)=={'src/workflow/stage1_open_action_value.py','src/workflow/intraday_recovery_direct_allocation.py'}
sealed_commit=src['sealed_source_commit']
for mismatch in mismatches:
 original=subprocess.check_output(['git','show',sealed_commit+':'+mismatch['path']],cwd=root)
 assert hashlib.sha256(original).hexdigest()==mismatch['sealed_sha256']
patch=subprocess.check_output(['git','diff',sealed_commit,'HEAD','--',*(x['path'] for x in mismatches)],cwd=root,text=True)
(card/'code_diff.patch').write_text(patch)
call_source=ast.parse((root/'src/schedule/stage1_regime_paper.py').read_text());paper_calls=[]
for node in ast.walk(call_source):
 if isinstance(node,ast.Call) and isinstance(node.func,ast.Name) and node.func.id=='prepare_on_policy_open':paper_calls.append([k.arg for k in node.keywords])
assert paper_calls and all('state_transition' not in x for x in paper_calls)
settings=_settings(root);nas=_latest_verified_snapshot(settings)
provider=(root/src['provider_uri']).resolve();qlib.init(provider_uri=str(provider),region='us')
calendar=pd.DatetimeIndex(D.calendar(start_time='2026-10-19',end_time='2026-10-23 16:00:00',freq='15min'))
flags=D.features(['SOXL','SOXX'],['$is_complete','$bar_count'],start_time='2026-10-19 09:45:00',end_time='2026-10-21 16:00:00',freq='15min').reset_index()
flags['date']=pd.to_datetime(flags['datetime']).dt.date;early=flags.groupby(['instrument','date'],sort=True,group_keys=False).head(4)
checks=[]
for (instrument,date),g in early.groupby(['instrument','date'],sort=True):
 checks.append({'instrument':str(instrument),'date':str(date),'first_four_rows':len(g),'complete_nonnull':int(g['$is_complete'].notna().sum()),'min_is_complete':None if g['$is_complete'].isna().all() else float(g['$is_complete'].min()),'bar_count_min':None if g['$bar_count'].isna().all() else float(g['$bar_count'].min())})
assert len(checks)==6
ps=subprocess.run(['launchctl','list'],capture_output=True,text=True,timeout=5)
launchd_matches=[line.strip() for line in ps.stdout.splitlines() if 'stage1' in line.lower() or 'soxl' in line.lower()][:15] if ps.returncode==0 else []
result={'status':'paper_forward_not_ready_release_hash_and_current_source_freshness','observed_at_utc':now.isoformat(),'release_manifest_sha256':src['release_manifest_sha256'],'sealed_source_commit':sealed_commit,'release_code_files':len(files),'code_mismatches':mismatches,'code_diff_sha256':sha(card/'code_diff.patch'),'paper_prepare_on_policy_open_keywords':paper_calls,'paper_passes_state_transition':False,'is_complete_branch_active_in_current_provider':'$is_complete' in provider_manifest.get('fields',[]),'historical_first_four_bar_checks':checks,'observed_first_four_bars_all_complete':all(x['first_four_rows']==4 and x['min_is_complete']==1 for x in checks),'provider_manifest_as_of':provider_manifest.get('as_of'),'Qlib_calendar_max':str(calendar.max()),'Qlib_calendar_dates':[str(x.date()) for x in calendar.normalize().unique()],'NAS_latest_verified_snapshot_id':nas.get('source_id'),'NAS_latest_verified_as_of':nas.get('as_of'),'NAS_latest_verified_at':nas.get('verified_at'),'NAS_frozen_snapshot_same':nas.get('source_id')==src['NAS_verified_snapshot_id_at_freeze'],'launchctl_read_exit':ps.returncode,'launchctl_matching_jobs':launchd_matches,'new_model_fits':0,'new_native_accounts':0,'market_outcome_reads':0,'release_writes':0,'orders':0,'interpretation':'Sealed code mismatch blocks runtime now. Optional transition default is unused in paper call. is_complete filtering is semantically active for future incomplete bars even if observed completed dates are all1. Current Qlib provider and NAS source as_of must be treated at observed time; no production reseal or forward-day claim.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','code_diff':'code_diff.patch','market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38915:MainThread](2026-10-23 09:05:11,804) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[38915:MainThread](2026-10-23 09:05:12,346) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[38915:MainThread](2026-10-23 09:05:12,347) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


{"status": "paper_forward_not_ready_release_hash_and_current_source_freshness", "observed_at_utc": "2026-10-23T01:05:05.842513+00:00", "release_manifest_sha256": "bd7210b4a907dd1da08cd26de21854be976a5cb0041f74e6d04b23972de83156", "sealed_source_commit": "9ff3fcf1243af6cc524090e64f7ab27302ef3c21", "release_code_files": 12, "code_mismatches": [{"path": "src/workflow/stage1_open_action_value.py", "sealed_sha256": "3b6cc0d1f385ff797bdcaaaa1aca599c554e9c91931861bb66777fb1d8f25359", "current_sha256": "6459fb44b1019bd8c4555ebd6b2ba6e9dbe8241844875b23a439837040734671"}, {"path": "src/workflow/intraday_recovery_direct_allocation.py", "sealed_sha256": "7973c6395d3232868f739f1fdc4251ce03552e6e10dd70126c8481c6d1b3c784", "current_sha256": "48b4a62a1c8139ecaa5d224023693bd4b1589851feab4b3a114d7188428cb013"}], "code_diff_sha256": "2e2621864fe111ee1980529d3cf2c70dcb2871e3fcd2e6a007ad6063809b8ecb", "paper_prepare_on_policy_open_keywords": [[null, "model", "action_spec"]], "paper_passes_state_transition"

# Stage1 纸面前向观察：代码身份与来源新鲜度

只读检查密封 release 的 **12** 份代码、当前 Git 原字节、已登记调度、Qlib 15 分钟日历与 NAS 最新已验证快照。当前两份代码 SHA-256 不符，精确补丁保存在 `code_diff.patch`。`stage1_open_action_value.py` 仅新增可选 `state_transition` 参数；纸面调用没有传入该参数，仍走原默认转移。`intraday_recovery_direct_allocation.py` 则在 provider 声明 `$is_complete` 时，只采用首四条都完整的首小时 bar，这是对未来不完整来源的**实际语义变化**，不能通过改 manifest 哈希跳过审核。

当前 provider 的确声明 `$is_complete`。用 Qlib `D.calendar`/`D.features` 对 10 月 19–21 日 SOXL/SOXX 共 **6 个标的日、24 条首小时 bar** 的限定检查，完整标志全为 1；这只说明这些已完成日的过滤是空操作，**不证明将来的部分 bar 与旧 release 等价**。本轮截至 **2026-10-23T01:05:05.842513+00:00**，该 Qlib provider 日历及 NAS 最新已验证 `us_semis_intraday` 快照都只到 **10 月 21 日**，尚不足以重建 10 月 22 日首小时。数据更新何时到达需按实际后续收据确认，不能从一次观察断言调度故障。

`launchctl` 有 `com.soxl.stage1.regime.paper` 登记；先前 10 月 22 日运行日志已记载 release 哈希拦截。此次没有启动、重启或更新纸面 release，也没有模型拟合、账户、订单、SOXX 成绩或合格前向日。要恢复未来观察，须分别完成：①对首小时完整性语义做独立审核并重新密封 release；②确保当次决策前可用的已验证分钟来源进入 Qlib provider；③按冻结前向计划确认双账户、时间与风险门。已有未通过的 Stage1 纸面基线即使恢复运行，也不能冒充新候选跑赢 SOXX。
